# Hi-EF Phase 2: reliability-gated residual inner-development matrix

This notebook runs the frozen v0.8 2×2 ablation: `ungated`, `gate`, `counterfactual`, and `gate_counterfactual`, for development seeds 42, 123, and 456. Every training and evaluation row comes from the original **train** partition. Original validation and test remain sealed.

Attach `ptrnghieu/hi-ef-features-v2`, enable a T4 GPU and Internet, then choose **Save Version → Save & Run All**.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO = Path('/kaggle/working/hi-ef-materials')
FEATURES = Path('/kaggle/input/datasets/ptrnghieu/hi-ef-features-v2')
OUTPUT = Path('/kaggle/working/reliability_inner_matrix')

if not (REPO / '.git').exists():
    subprocess.run([
        'git', 'clone', '--branch', 'experiments', '--single-branch',
        'https://github.com/ptrnghieu/hi-ef-materials.git', str(REPO)
    ], check=True)
else:
    subprocess.run([
        'git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'experiments'
    ], check=True)

assert (FEATURES / '01_00059.pt').is_file(), 'Feature dataset is not attached'
MANIFEST = REPO / 'experiments/manifests/inner_development_seed8042.csv'
SPEC = REPO / 'experiments/RESEARCH_SPEC_v0.8.md'
assert MANIFEST.is_file() and SPEC.is_file()
commit = subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True
).strip()
print('Ready at commit:', commit)

In [ ]:
# This CPU-safe test proves the committed inner split contains original-train rows only.
subprocess.run([
    'python', '-m', 'unittest', 'experiments/test_inner_development_split.py'
], cwd=REPO, env={**os.environ, 'PYTHONPATH': str(REPO / 'experiments')}, check=True)

In [ ]:
# The matrix runner first executes all PyTorch forward/backward contract tests.
command = [
    'python', str(REPO / 'experiments/run_reliability_inner_matrix.py'),
    '--manifest', str(MANIFEST),
    '--features-dir', str(FEATURES),
    '--output-dir', str(OUTPUT),
]
subprocess.run(command, check=True)

In [ ]:
import json
import pandas as pd

summary_path = OUTPUT / 'reliability_inner_matrix_summary.json'
table_path = OUTPUT / 'reliability_inner_matrix.csv'
summary = json.loads(summary_path.read_text())
assert summary['original_validation_evaluated'] is False
assert summary['test_evaluated'] is False
assert summary['partitions_touched'] == [
    'original-train/inner-train', 'original-train/inner-development'
]
assert len(summary['runs']) == 3
assert all(len(runs) == 4 for runs in summary['runs'].values())
display(pd.read_csv(table_path))
print(json.dumps(summary['aggregates'], indent=2))
print(json.dumps(summary['advancement_gate'], indent=2))
print('Download:', summary_path)
print('Download:', table_path)